In [1]:
import pandas as pd

# =========================
# 1. Définition des groupes
# =========================
G7 = [
    "USA",
    "France",
    "Germany",
    "Italy",
    "United Kingdom",
    "Canada",
    "Japan"
]

BRICS = [
    "Brazil",
    "Russian Federation",
    "India",
    "China",
    "South Africa"
]

def assign_group(country):
    if country in G7:
        return 1
    elif country in BRICS:
        return 2
    else:
        return 3


# =========================
# 2. Fichiers à fusionner
# =========================
files_map = {
    "GDP_per_capita.csv": "GDP_per_capita",
    "GINI_Index.csv": "Gini_Index",
    "Top_10%_incomesharehaut.csv": "Top_10_Income_Share",
    "Top_10%_wealthsharehaut.csv": "Top_10_Wealth_Share",
    "Top_50%_incomesharebas.csv": "Bottom_50_Income_Share",
    "Top_50%_wealthsharebas.csv": "Bottom_50_Wealth_Share",
    "Total_population.csv": "Total_Population"
}

def clean_country_name(col):
    return col.split("\n")[-1].strip()


# =========================
# 3. Fusion des fichiers
# =========================
merged_df = None

for filename, var_name in files_map.items():
    df = pd.read_csv(f"../data/{filename}", sep=";", skiprows=1)
    df.columns = [c.strip() for c in df.columns]

    countries = [c for c in df.columns if c not in ["Year", "Percentile"]]

    df_long = df.melt(
        id_vars=["Year"],
        value_vars=countries,
        var_name="Country",
        value_name=var_name
    )

    df_long["Country"] = df_long["Country"].apply(clean_country_name)

    if merged_df is None:
        merged_df = df_long
    else:
        merged_df = pd.merge(
            merged_df,
            df_long,
            on=["Year", "Country"],
            how="outer"
        )


# =========================
# 4. Ajout du groupe
# =========================
merged_df["Group"] = merged_df["Country"].apply(assign_group)

merged_df = merged_df.sort_values(["Country", "Year"])


# =========================
# 5. Sauvegarde
# =========================
merged_df.to_csv("../data/combined_data.csv", index=False)

merged_df.head()


,Year,Country,GDP_per_capita,Gini_Index,Top_10_Income_Share,Top_10_Wealth_Share,Bottom_50_Income_Share,Bottom_50_Wealth_Share,Total_Population,Group
0,1980,Bangladesh,2626.4911,0.4732,0.3652,0.5731,0.1997,0.0491,88016432,3
18,1981,Bangladesh,2613.7059,0.4732,0.3652,0.5731,0.1997,0.0491,90303105,3
36,1982,Bangladesh,2619.9257,0.4732,0.3652,0.5731,0.1997,0.0491,92814507,3
54,1983,Bangladesh,2679.1805,0.4732,0.3652,0.5731,0.1997,0.0491,95335155,3
72,1984,Bangladesh,2710.5999,0.4836,0.3774,0.5762,0.1947,0.0486,97814966,3
